In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("hr.csv")

In [ ]:
# Hacemos una copia del dataset original
# MUY IMPORTANTE — nunca trabajamos sobre el original
df_limpio = df.copy()

print("Copia creada correctamente!")
print("Filas:", df_limpio.shape[0])
print("Columnas:", df_limpio.shape[1])

¿Por qué df.copy()?
Sin .copy() cualquier cambio en df_clean también cambiaría df — Python compartiría la misma tabla en memoria. Con .copy() son dos tablas completamente independientes 😊

# TAREA A1 — Normalizar el texto de JobRole

In [4]:
df_clean = df.copy()

In [ ]:
# TAREA A1 — Normalizar el texto de JobRole

# ── Antes de limpiar — ver cómo está ─────────────
print("ANTES de limpiar:")
print(df_clean["JobRole"].unique())

# ── La función de limpieza ────────────────────────
def fix_job_role(df):
    df["JobRole"] = df["JobRole"].str.strip()
    df["JobRole"] = df["JobRole"].str.title()
    return df

# ── Aplicar la función ────────────────────────────
df_clean = fix_job_role(df_clean)

# ── Después de limpiar — comprobar resultado ──────
print("\nDESPUES de limpiar:")
print(df_clean["JobRole"].unique())

✅ Lo que hicimos — la solución:
Con solo dos líneas dentro de la función:

.str.strip() → eliminó los espacios
.str.title() → corrigió las mayúsculas

' sALES eXECUTIVE '  →  'Sales Executive'
' rESEARCH sCIENTIST '  →  'Research Scientist'

💡 Conclusión para la demo:

"La columna JobRole tenía un problema de formato en los 9 roles — espacios y mayúsculas incorrectas. Lo normalizamos con strip() y title(). Esto es importante porque sin esta limpieza los gráficos de la Fase 3 mostrarían roles duplicados o no encontrarían los datos correctamente"

### Conclusiones Tarea A1 — JobRole limpio ✅
- Problema: texto corrupto en los 9 roles
  → espacios al principio y final
  → mayúsculas mezcladas aleatoriamente
- Impacto: sin limpiar, Python trataría
  'Sales Executive' y ' sALES eXECUTIVE '
  como dos categorías distintas
- Solución: .str.strip() + .str.title()
- Resultado: 9 roles con formato correcto
  y listos para graficar en Fase 3

# TAREA A2 — Convertir tipos de datos

In [ ]:
def fix_types(df):                                               
    cols = ["Age", "JobSatisfaction", "MonthlyIncome", "YearsWithCurrManager"]                       
    for col in cols:                                                                                
        df[col] = df[col].fillna(0) 
        df[col] = df[col].astype(int)
    return df

df_clean = fix_types(df_clean)
print("Tipos convertidos!")

🔢 TAREA A2 — Convertir tipos de datos
¿Por qué hay que hacerlo?
En la Fase 1 detectamos que estas columnas eran float64 cuando deberían ser enteras:

Age → 41.0 debería ser 41
JobSatisfaction → 3.0 debería ser 3
MonthlyIncome → 4907.0 debería ser 4907
YearsWithCurrManager → 5.0 debería ser 5

¿Por qué son float? Porque tienen nulos — pandas convierte automáticamente a float cuando hay NaN en una columna de enteros 😊

Definimos una función que recibe un DataFrame y lo devuelve con los tipos corregidos             
Lista de las 4 columnas que hay que convertir                           
Recorremos cada columna de la lista una por una                              
Paso 1 — rellenamos los nulos con 0 porque Python no puede convertir NaN a entero directamente:        df[col] = df[col].fillna(0) 

41.0 → sigue siendo 41.0
NaN → se convierte en 0.0

Paso 2 — convertimos a entero:                                                                          df[col] = df[col].astype(int)

41.0 → 41 ✅
0.0 → 0 ← los nulos quedan como 0 de momento
Devolvemos el DataFrame ya corregido — sin este return la función no sirve de nada                       return df
Aplicamos la función y guardamos el resultado en df_clean                                              df_clean = fix_types(df_clean)

⚠️ Nota importante:
Los 0 que pusimos donde había nulos son temporales — en la Tarea A3 los sustituiremos por la mediana real de cada columna 😊

### Conclusiones Tarea A2 — Tipos corregidos ✅

- Problema: Age, JobSatisfaction, MonthlyIncome
  y YearsWithCurrManager eran float64
  → tenían decimales innecesarios (41.0, 3.0...)
- Causa: pandas convierte a float cuando hay nulos
- Solución: fillna(0) + astype(int)
- Resultado: las 4 columnas son ahora int64
- Nota: los 0 de los nulos se corrigen en Tarea A3


In [ ]:
# Verificación — ver que los tipos cambiaron
cols_check = ["Age", "JobSatisfaction", "MonthlyIncome", "YearsWithCurrManager"]

for col in cols_check:
    print(f"{col}: {df_clean[col].dtype}")

print("\nPrimeros valores de Age:")
print(df_clean["Age"].head())

# TAREA A3 — Tratar los valores nulos

In [ ]:
# TAREA A3 — Tratar los valores nulos

# ── Antes — ver cuántos nulos quedan ─────────────
print("ANTES de tratar nulos:")
nulos_antes = df_clean.isnull().sum()
print(nulos_antes[nulos_antes > 0])

# ── La función ────────────────────────────────────
def fix_nulls(df):

    # Columnas NUMÉRICAS → rellenar con la MEDIANA
    cols_mediana = ["YearsWithCurrManager", "TrainingTimesLastYear",
                    "Age", "JobSatisfaction", "MonthlyIncome"]
    for col in cols_mediana:
        mediana = df[col].median()
        df[col] = df[col].replace(0, mediana)
        print(f"  {col}: nulos rellenados con mediana ({mediana})")

    # Columnas de TEXTO → rellenar con la MODA
    cols_moda = ["MaritalStatus", "BusinessTravel",
                 "EducationField", "OverTime", "Department"]
    for col in cols_moda:
        moda = df[col].mode()[0]
        df[col] = df[col].fillna(moda)
        print(f"  {col}: nulos rellenados con moda ('{moda}')")

    # TrainingTimesLastYear por separado
    df["TrainingTimesLastYear"] = df["TrainingTimesLastYear"].fillna(
        df["TrainingTimesLastYear"].median()).astype(int)

    return df

# ── Aplicar la función ────────────────────────────
print("\nRellenando nulos:")
df_clean = fix_nulls(df_clean)

# ── Después — comprobar que no quedan nulos ───────
print("\nDESPUES de tratar nulos:")
nulos_despues = df_clean.isnull().sum()
nulos_despues = nulos_despues[nulos_despues > 0]
if len(nulos_despues) == 0:
    print("  No quedan nulos!")
else:
    print(nulos_despues)

### Conclusiones Tarea A3 — Nulos tratados ✅
- Columnas numéricas → rellenadas con MEDIANA
  (más robusta que la media ante valores extremos)
    - YearsWithCurrManager → 2
    - TrainingTimesLastYear → 3
    - Age → 35
    - JobSatisfaction → 3
    - MonthlyIncome → 4876
- Columnas de texto → rellenadas con MODA
  (el valor más frecuente)
    - MaritalStatus → 'Married'
    - BusinessTravel → 'Travel_Rarely'
    - EducationField → 'Life Sciences'
    - OverTime → 'No'
    - Department → 'Research & Development'
- StandardHours → pendiente eliminar en Tarea B1

In [8]:
df_clean.to_csv('Fase2_ParejaA_limpieza.csv', index=False)